So, for the model training, I decided to use a gradient boost decision tree model, because they can capture complex relationship between differnt columns, 

They can't capture the effect of sequence on time series. However, this isnt an issue if you calculate time-based feature for them, like I did in the feature engineering file.

**NOTE:** I run the code on Kaggle because it has GPU support, which makes training much faster.

First, I tried to use RMSE as an evaluation metric.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

spark = SparkSession.builder.appName("SalesPrediction").getOrCreate()

data_path = "/kaggle/input/grocery-sales-training/part-00000-e17247d3-2ceb-4ccd-9675-f24e3aa2a2e8-c000.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

features = ['onpromotion', 'family', 'class', 'perishable', 'city', 'state', 
            'type', 'cluster', 'transactions', 'dcoilwtico', 'holiday', 
            'day_of_week', 'month', 'day_of_month', 
            'promo_last_7_days','promo_last_14_days','promo_last_30_days',
            'sales_lag_1','sales_lag_7','sales_lag_14','sales_lag_30',
            'weekly_sales_avg','rolling_7_mean','rolling_14_mean','rolling_30_mean',
            'perishable_rolling_7_mean']

target = 'unit_sales'


# Had to sample the data since the full dataset is too large
df_sample = df.sample(fraction=0.1)

# I split the data into training and validation sets based on date
train_df = df_sample.filter((F.col("date") >= "2015-01-01") & (F.col("date") < "2016-08-01"))
valid_df = df_sample.filter(F.col("date") >= "2016-08-01")

train_pd = train_df.select(features + [target]).toPandas()
valid_pd = valid_df.select(features + [target]).toPandas()

categorical_features = ['family', 'class', 'city', 'state', 'type', 'cluster', 'day_of_week', 'month']

for cat in categorical_features:
    train_pd[cat] = train_pd[cat].astype('category')
    valid_pd[cat] = valid_pd[cat].astype('category')

train_weight = train_pd['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0)
valid_weight = valid_pd['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0)

train_data = lgb.Dataset(train_pd[features], label=train_pd[target],
                         weight=train_weight, categorical_feature=categorical_features)
valid_data = lgb.Dataset(valid_pd[features], label=valid_pd[target],
                         weight=valid_weight, categorical_feature=categorical_features)

# I also experimented with different parameters to see which one worked the best, and this is what I came up with
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 80,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=3000,
    valid_sets=[train_data, valid_data],
    valid_names=['train','valid']
)

y_pred = model.predict(valid_pd[features], num_iteration=model.best_iteration)
rmse = np.sqrt(mean_squared_error(valid_pd[target], y_pred))
print("Validation RMSE:", rmse)

model.save_model('model3.txt')

Validation RMSE: 13.22764625505616


The RMSE came out quite high, but this is because there are some extreme outliers in the data, so I changed to RMSLE, which isn't affected by the outlier as much as RMSE, (RMSLE cant be used with negative value but we can just set the negative value to 0, which wont affect much since there is very little negative value).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import lightgbm as lgb
import numpy as np

spark = SparkSession.builder.appName("SalesPrediction").getOrCreate()

data_path = "/kaggle/input/grocery-sales-training/part-00000-e17247d3-2ceb-4ccd-9675-f24e3aa2a2e8-c000.csv"
df = spark.read.csv(data_path, header=True, inferSchema=True)

features = ['onpromotion', 'family', 'class', 'perishable', 'city', 'state', 
            'type', 'cluster', 'transactions', 'dcoilwtico', 'holiday', 
            'day_of_week', 'month', 'day_of_month', 
            'promo_last_7_days','promo_last_14_days','promo_last_30_days',
            'sales_lag_1','sales_lag_7','sales_lag_14','sales_lag_30',
            'weekly_sales_avg','rolling_7_mean','rolling_14_mean','rolling_30_mean',
            'perishable_rolling_7_mean']

target = 'unit_sales'

df_sample = df.sample(fraction=0.1)

train_df = df_sample.filter((F.col("date") >= "2015-01-01") & (F.col("date") < "2016-08-01"))
valid_df = df_sample.filter(F.col("date") >= "2016-08-01")

train_pd = train_df.select(features + [target]).toPandas()
valid_pd = valid_df.select(features + [target]).toPandas()

categorical_features = ['family', 'class', 'city', 'state', 'type', 'cluster', 'day_of_week', 'month']

for cat in categorical_features:
    train_pd[cat] = train_pd[cat].astype('category')
    valid_pd[cat] = valid_pd[cat].astype('category')

train_weight = train_pd['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0)
valid_weight = valid_pd['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0)

train_data = lgb.Dataset(train_pd[features], label=train_pd[target], 
                         weight=train_weight, categorical_feature=categorical_features)
valid_data = lgb.Dataset(valid_pd[features], label=valid_pd[target], 
                         weight=valid_weight, categorical_feature=categorical_features)

params = {
    'objective': 'regression',
    'metric': 'None',
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'num_leaves': 80,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=3000,
    valid_sets=[train_data, valid_data],
    valid_names=['train','valid']
)

y_pred = model.predict(valid_pd[features], num_iteration=model.best_iteration)

y_pred_clip = np.clip(y_pred, 0, None)
y_true_clip = np.clip(valid_pd[target], 0, None)

weighted_log_error = (valid_weight * (np.log1p(y_pred_clip) - np.log1p(y_true_clip))**2).sum() / valid_weight.sum()
weighted_rmsle = np.sqrt(weighted_log_error)

print("Validation RMSLE:", weighted_rmsle)

model.save_model('model4.txt')

Validation RMSLE: 0.5347119559878581

The RMSLE came out to be about 0.53, when looking at e^(0.53) - 1, its about 70% error, which is still not great, but since the data had so many outlier this is probably acceptable.

Now, lets test the model with the test set, using both RMSE and RMSLE.

For RMSE first,

In [ ]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import lightgbm as lgb
import numpy as np

spark = SparkSession.builder.appName("Testing").getOrCreate()

test_path = "/kaggle/input/grocery-sales-testing/part-00000-14b01283-c3b2-465c-badd-4692dedb0551-c000.csv"
df_test = spark.read.csv(test_path, header=True, inferSchema=True)

features = ['onpromotion', 'family', 'class', 'perishable', 'city', 'state', 
            'type', 'cluster', 'transactions', 'dcoilwtico', 'holiday', 
            'day_of_week', 'month', 'day_of_month', 
            'promo_last_7_days','promo_last_14_days','promo_last_30_days',
            'sales_lag_1','sales_lag_7','sales_lag_14','sales_lag_30',
            'weekly_sales_avg','rolling_7_mean','rolling_14_mean','rolling_30_mean',
            'perishable_rolling_7_mean']

categorical_features = ['family', 'class', 'city', 'state', 'type', 'cluster', 'day_of_week', 'month']

model = lgb.Booster(model_file='/kaggle/input/grocery-model3/other/default/1/model3.txt')

# Take a sample since I can't process the full test set
df_sample = df_test.sample(withReplacement=False, fraction=0.2, seed=42)

columns_needed = features + ['id']

if 'unit_sales' in df_test.columns:
    columns_needed.append('unit_sales')

pdf_sample = df_sample.select(columns_needed).toPandas()

for cat in categorical_features:
    pdf_sample[cat] = pdf_sample[cat].astype('category')

y_sample_pred = model.predict(pdf_sample[features], num_iteration=model.best_iteration)

def weighted_rmse(y_true, y_pred, perishable):
    weights = np.where(perishable == 1, 1.25, 1.0)
    
    squared_errors = (y_true - y_pred) ** 2
    weighted_squared_errors = squared_errors * weights
    
    wmse = np.sum(weighted_squared_errors) / np.sum(weights)
    return np.sqrt(wmse)

if 'unit_sales' in pdf_sample.columns:
    test_unit_sales = pdf_sample['unit_sales']
    perishable_values = pdf_sample['perishable']
    
    wrmse_sample = weighted_rmse(test_unit_sales, y_sample_pred, perishable_values)
    
    print("Weighted RMSE:", wrmse_sample)

predictions = pd.DataFrame({
    'id': pdf_sample['id'],
    'unit_sales': y_sample_pred
})

predictions.to_csv('rmse_prediction.csv', index=False, float_format='%.4f')

Weighted RMSE: 15.581640203015498

In [ ]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd
import lightgbm as lgb
import numpy as np

spark = SparkSession.builder.appName("RMSLE_Testing").getOrCreate()

test_path = "/kaggle/input/grocery-sales-testing/part-00000-14b01283-c3b2-465c-badd-4692dedb0551-c000.csv"
df_test = spark.read.csv(test_path, header=True, inferSchema=True)

features = ['onpromotion', 'family', 'class', 'perishable', 'city', 'state', 
            'type', 'cluster', 'transactions', 'dcoilwtico', 'holiday', 
            'day_of_week', 'month', 'day_of_month', 
            'promo_last_7_days','promo_last_14_days','promo_last_30_days',
            'sales_lag_1','sales_lag_7','sales_lag_14','sales_lag_30',
            'weekly_sales_avg','rolling_7_mean','rolling_14_mean','rolling_30_mean',
            'perishable_rolling_7_mean']

categorical_features = ['family', 'class', 'city', 'state', 'type', 'cluster', 'day_of_week', 'month']

model = lgb.Booster(model_file='/kaggle/input/grocery-model4/other/default/1/model4.txt')

df_sample = df_test.sample(withReplacement=False, fraction=0.2, seed=42)

columns_needed = features + ['id']
if 'unit_sales' in df_test.columns:
    columns_needed.append('unit_sales')

pdf_sample = df_sample.select(columns_needed).toPandas()

for cat in categorical_features:
    pdf_sample[cat] = pdf_sample[cat].astype('category')

y_sample_pred = model.predict(pdf_sample[features], num_iteration=model.best_iteration)

def weighted_rmsle(y_true, y_pred, perishable):
    weights = np.where(perishable == 1, 1.25, 1.0)
    
    y_true_adj = np.maximum(y_true, 0)
    y_pred_adj = np.maximum(y_pred, 0)
    
    log_errors = np.log1p(y_true_adj) - np.log1p(y_pred_adj)
    squared_log_errors = log_errors ** 2
    weighted_squared_log_errors = squared_log_errors * weights
    
    wmsle = np.sum(weighted_squared_log_errors) / np.sum(weights)
    
    return np.sqrt(wmsle)


test_unit_sales = pdf_sample['unit_sales']
perishable_values = pdf_sample['perishable']

wrmsle_sample = weighted_rmsle(test_unit_sales, y_sample_pred, perishable_values)

print("Weighted RMSLE:", wrmsle_sample)

predictions = pd.DataFrame({
    'id': pdf_sample['id'],
    'unit_sales': y_sample_pred
})

predictions.to_csv('rmsle_predictions.csv', index=False, float_format='%.4f')

Sample Test Weighted RMSLE: 0.5135112226080921

The RMSE of the test set came out to be about 15.5, while the RMSLE is about 0.51 (or aroun 65%). Given the range of the unit sales, this is probably an ok model that can be further improve upon by doing better feature engineering and hyperparameters tuning.